# MyNextBook — Data Cleaning & EDA

**Pipeline:** `python -m src.pipeline` → `data/raw/books.csv`  
**This notebook:** loads that CSV, applies cleaning, and does thorough EDA.

Sections:
1. Load raw data from CSV
2. Missing value analysis (before cleaning)
3. Data cleaning — step by step with before/after counts
4. EDA — distributions (ratings, pages, years, description length)
5. Category analysis
6. Text analysis (top words per genre)
7. Temporal analysis (books per decade)
8. Recommender model walkthrough
9. Method comparison (content vs SVD vs hybrid)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.figsize': (11, 4.5), 'figure.dpi': 110})

CSV_PATH = '../data/raw/books.csv'

## 1. Load Raw Data

In [ ]:
from src.data_loader import load_raw

raw_books = load_raw(CSV_PATH)
df_raw = pd.DataFrame(raw_books)
df_raw['author_str']   = df_raw['authors'].apply(lambda a: ', '.join(a) if a else 'Unknown')
df_raw['category_str'] = df_raw['categories'].apply(lambda c: c[0] if c else '')
df_raw['desc_length']  = df_raw['description'].str.len()

print(f'Raw books loaded : {len(df_raw):,}')
print(f'Columns          : {list(df_raw.columns)}')
df_raw[['title','author_str','category_str','average_rating','ratings_count','desc_length','language']].head(8)

## 2. Missing Value Analysis (Before Cleaning)

In [ ]:
check_cols = ['title','description','authors','categories','average_rating',
              'ratings_count','page_count','published_date','thumbnail','language']

missing = pd.DataFrame({
    'missing_count': df_raw[check_cols].isnull().sum(),
    'empty_list'   : df_raw[check_cols].apply(
        lambda col: col.apply(lambda v: isinstance(v, list) and len(v) == 0).sum()
    ),
    'empty_string' : df_raw[check_cols].apply(
        lambda col: col.apply(lambda v: v == '').sum()
    ),
})
missing['total_missing'] = missing.sum(axis=1)
missing['pct_missing']   = (missing['total_missing'] / len(df_raw) * 100).round(1)
missing.sort_values('pct_missing', ascending=False)

In [ ]:
# Visualise missingness
fig, ax = plt.subplots(figsize=(9, 4))
missing['pct_missing'].sort_values().plot(kind='barh', ax=ax, color='tomato')
ax.set_xlabel('% missing or empty')
ax.set_title('Missing / Empty Data per Field (raw data)')
ax.axvline(5, color='grey', linestyle='--', linewidth=0.8, label='5% line')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Language breakdown
lang_counts = df_raw['language'].value_counts().head(10)
print('Top languages in raw data:')
print(lang_counts.to_string())
non_en_pct = (df_raw['language'] != 'en').mean() * 100
print(f'\nNon-English books: {non_en_pct:.1f}%')

## 3. Data Cleaning — Step by Step

In [ ]:
from src.cleaning import clean_books

print('Cleaning steps and books removed at each step:\n')
cleaned_books, report = clean_books(raw_books, verbose=True)

In [ ]:
# Waterfall chart — impact of each cleaning step
steps = [k for k in report if k not in ('raw','final','nulled_bad_ratings','bad_ratings_nulled')]
values = [report[s] for s in steps]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(steps, values, color='steelblue', edgecolor='white')
ax.bar_label(bars, fmt='%d', padding=3)
ax.set_ylabel('Books removed')
ax.set_title(f'Cleaning Impact  |  Raw: {report["raw"]:,}  →  Clean: {report["final"]:,}')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
df = pd.DataFrame(cleaned_books)
df['desc_length'] = df['description'].str.len()
df['category_str'] = df['categories'].apply(lambda c: c[0] if c else 'Unknown')
df['author_str']   = df['authors'].apply(lambda a: a[0] if a else 'Unknown')

print(f'Clean dataset: {len(df):,} books')
df[['title','author_str','category_str','average_rating','pub_year','desc_length']].head(8)

## 4. Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# 4a — Rating distribution
rated = df.dropna(subset=['average_rating'])
axes[0,0].hist(rated['average_rating'], bins=20, color='steelblue', edgecolor='white')
axes[0,0].set_title(f'Rating distribution  (n={len(rated):,})')
axes[0,0].set_xlabel('Average rating (1–5)')
axes[0,0].axvline(rated['average_rating'].mean(), color='red', linestyle='--',
                   label=f'mean={rated["average_rating"].mean():.2f}')
axes[0,0].legend()

# 4b — Ratings count (log scale)
rc = df[df['ratings_count'] > 0]
axes[0,1].hist(np.log10(rc['ratings_count']+1), bins=30, color='coral', edgecolor='white')
axes[0,1].set_title('Ratings count (log₁₀ scale)')
axes[0,1].set_xlabel('log₁₀(ratings_count)')
axes[0,1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'10^{x:.0f}'))

# 4c — Page count
paged = df.dropna(subset=['page_count'])
axes[1,0].hist(paged['page_count'].clip(upper=800), bins=30, color='mediumseagreen', edgecolor='white')
axes[1,0].set_title(f'Page count  (n={len(paged):,}, clipped at 800)')
axes[1,0].set_xlabel('Pages')

# 4d — Description length
axes[1,1].hist(df['desc_length'].clip(upper=2500), bins=30, color='mediumpurple', edgecolor='white')
axes[1,1].set_title('Description length (chars, clipped at 2500)')
axes[1,1].set_xlabel('Characters')

plt.tight_layout()
plt.show()

In [ ]:
# Rating vs popularity scatter (coloured by top-6 categories)
top6 = df['category_str'].value_counts().head(6).index
df_plot = df[df['category_str'].isin(top6) & df['ratings_count'].gt(0)].dropna(subset=['average_rating'])

fig, ax = plt.subplots(figsize=(9, 5))
for cat, grp in df_plot.groupby('category_str'):
    ax.scatter(np.log10(grp['ratings_count']+1), grp['average_rating'],
               alpha=0.4, label=cat, s=18)
ax.set_xlabel('log₁₀(ratings count)')
ax.set_ylabel('Average rating')
ax.set_title('Popularity vs Quality by Genre')
ax.legend(bbox_to_anchor=(1,1))
plt.tight_layout()
plt.show()

## 5. Category Analysis

In [ ]:
all_cats = [cat for cats in df['categories'] for cat in cats]
top_cats = Counter(all_cats).most_common(20)
cats_df  = pd.DataFrame(top_cats, columns=['Category','Count'])

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=cats_df, x='Count', y='Category', palette='Blues_r', ax=ax)
ax.set_title('Top 20 Normalised Categories')
plt.tight_layout()
plt.show()

In [ ]:
# Average rating per top-15 category
cat_ratings = []
for cat in [c for c,_ in Counter(all_cats).most_common(15)]:
    books_in_cat = df[df['categories'].apply(lambda cs: cat in cs)]
    rated_in_cat = books_in_cat.dropna(subset=['average_rating'])
    if len(rated_in_cat) >= 5:
        cat_ratings.append({'category': cat, 'mean_rating': rated_in_cat['average_rating'].mean(),
                            'count': len(rated_in_cat)})

cat_rat_df = pd.DataFrame(cat_ratings).sort_values('mean_rating', ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
bars = sns.barplot(data=cat_rat_df, x='mean_rating', y='category', palette='RdYlGn', ax=ax)
ax.set_xlim(3.0, 5.0)
ax.set_title('Mean Rating per Genre (min 5 rated books)')
plt.tight_layout()
plt.show()

## 6. Text Analysis — Top Words per Genre

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

focus_genres = ['Science Fiction','Fantasy','Mystery','Biography','Self-Help','History']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

vec = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1,1))
vec.fit(df['description'])
feature_names = np.array(vec.get_feature_names_out())

for ax, genre in zip(axes, focus_genres):
    mask = df['categories'].apply(lambda cs: genre in cs)
    subset_desc = df.loc[mask, 'description']
    if len(subset_desc) < 5:
        ax.set_visible(False)
        continue
    tfidf_sub = vec.transform(subset_desc).toarray()
    mean_scores = tfidf_sub.mean(axis=0)
    top_idx = mean_scores.argsort()[-12:][::-1]
    top_words = feature_names[top_idx]
    top_scores = mean_scores[top_idx]
    ax.barh(top_words[::-1], top_scores[::-1], color='steelblue')
    ax.set_title(f'{genre}  (n={mask.sum():,})')
    ax.set_xlabel('Mean TF-IDF')

plt.suptitle('Top Descriptive Words per Genre (TF-IDF)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 7. Temporal Analysis

In [ ]:
dated = df.dropna(subset=['pub_year'])
dated = dated[(dated['pub_year'] >= 1900) & (dated['pub_year'] <= 2025)].copy()
dated['decade'] = (dated['pub_year'] // 10 * 10).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Books per decade
decade_counts = dated['decade'].value_counts().sort_index()
axes[0].bar(decade_counts.index, decade_counts.values, width=8, color='steelblue', edgecolor='white')
axes[0].set_title('Books Published per Decade')
axes[0].set_xlabel('Decade')
axes[0].set_ylabel('Count')

# Mean rating per decade
decade_rating = dated.dropna(subset=['average_rating']).groupby('decade')['average_rating'].mean()
axes[1].plot(decade_rating.index, decade_rating.values, marker='o', color='coral')
axes[1].set_title('Mean Rating by Publication Decade')
axes[1].set_xlabel('Decade')
axes[1].set_ylabel('Average rating')
axes[1].set_ylim(3.0, 5.0)

plt.tight_layout()
plt.show()

## 8. Recommender Walkthrough

In [ ]:
from src.models import TFIDFRecommender, SentenceTransformerRecommender, HybridRecommender

# Pick 3 liked books from the cleaned corpus
liked_titles = ['Dune', 'Foundation', 'The Left Hand of Darkness']
liked = [b for b in cleaned_books if any(t.lower() in b['title'].lower() for t in liked_titles)]
if not liked:
    liked = [b for b in cleaned_books if 'Science Fiction' in b.get('categories', [])][:3]

print('Liked books:')
for b in liked:
    print(f"  • {b['title']}  ({', '.join(b.get('categories',[])[:2])})")


In [ ]:
liked_ids = {b['id'] for b in liked}
candidates = [b for b in cleaned_books if b['id'] not in liked_ids]

print(f'Fitting hybrid recommender on {len(cleaned_books):,} books…')
hybrid = HybridRecommender(alpha=0.5)  # 50/50 TF-IDF vs Sentence Transformer
hybrid.fit(cleaned_books)
recs = hybrid.recommend(liked, candidates, top_n=10)

recs_df = pd.DataFrame(recs)[['title','categories','average_rating','hybrid_score','tfidf_score','st_score']]
recs_df['categories'] = recs_df['categories'].apply(lambda c: ', '.join(c[:2]))
recs_df.index += 1
recs_df


In [ ]:
# Score breakdown bar chart
rp = pd.DataFrame(recs).head(10).copy()
rp['short_title'] = rp['title'].str[:28]
x, w = range(len(rp)), 0.28

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar([i-w for i in x], rp['tfidf_score'], w, label='TF-IDF',               color='steelblue')
ax.bar(list(x),          rp['st_score'],    w, label='Sentence Transformer',  color='coral')
ax.bar([i+w for i in x], rp['hybrid_score'],w, label='Hybrid (α=0.5)',        color='seagreen', alpha=0.85)
ax.set_xticks(list(x))
ax.set_xticklabels(rp['short_title'], rotation=35, ha='right')
ax.set_ylabel('Normalised similarity score')
ax.set_title('Top-10 Recommendations: TF-IDF vs Sentence Transformer vs Hybrid')
ax.legend()
plt.tight_layout()
plt.show()


## 9. Method Comparison (Precision, NDCG, Diversity)

In [ ]:
import numpy as np
from src.evaluation import diversity

def evaluate(rec_ids, relevant_ids, recs, books, k=10):
    hits = [r for r in rec_ids[:k] if r in relevant_ids]
    precision = len(hits) / k
    ndcg = sum(1.0 / np.log2(i + 2) for i, r in enumerate(rec_ids[:k]) if r in relevant_ids)
    ideal = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant_ids), k)))
    return {
        'precision@10': round(precision, 3),
        'ndcg@10':      round(ndcg / ideal if ideal else 0.0, 3),
        'diversity':    round(diversity(recs), 3),
    }

# Ground truth: books from the same genre as liked books
liked_genres = {cat for b in liked for cat in b.get('categories', [])}
relevant_ids = {
    b['id'] for b in cleaned_books
    if set(b.get('categories', [])) & liked_genres and b['id'] not in liked_ids
}
print(f'Relevant books in corpus (same genre): {len(relevant_ids):,}')


In [ ]:
# TF-IDF only
tfidf_model = TFIDFRecommender()
tfidf_model.fit(cleaned_books)
tfidf_recs = tfidf_model.recommend(liked, candidates, top_n=10)

# Sentence Transformer only
st_model = SentenceTransformerRecommender()
st_model.fit(cleaned_books)
st_recs = st_model.recommend(liked, candidates, top_n=10)

results = {}
for name, r in [('TF-IDF', tfidf_recs), ('Sentence Transformer', st_recs), ('Hybrid (α=0.5)', recs)]:
    m = evaluate([x['id'] for x in r], relevant_ids, r, cleaned_books, k=10)
    results[name] = m

comparison = pd.DataFrame(results).T
comparison


In [ ]:
comparison.plot(kind='bar', figsize=(9, 4), colormap='Set2', edgecolor='white')
plt.xticks(rotation=0)
plt.ylabel('Score')
plt.title('Method Comparison @ k=10')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

print('\nSweep alpha from 0 (pure TF-IDF) to 1 (pure Sentence Transformer):')
for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
    h = HybridRecommender(alpha=alpha)
    h.fit(cleaned_books)
    r = h.recommend(liked, candidates, top_n=10)
    m = evaluate([x['id'] for x in r], relevant_ids, r, cleaned_books, k=10)
    print(f'  alpha={alpha:.2f}  precision@10={m["precision@10"]:.3f}  ndcg@10={m["ndcg@10"]:.3f}  diversity={m["diversity"]:.3f}')
